# Лабораторная работа №3
## «Классификация временных рядов и прогнозирование на примере метеоданных шести городов России»

**Данные:** open-meteo, 6 городов, 2019–2025, файлы вида `{Город}_{год}-01-01_{год}-12-31.parquet`

| Город | Тип климата |
|-------|-------------|
| Москва | Умеренно-континентальный |
| Санкт-Петербург | Умеренно-континентальный |
| Сочи | Субтропический |
| Геленджик | Субтропический |
| Благовещенск | Резко-континентальный |
| Находка | Резко-континентальный |

## 0. Импорт библиотек и загрузка данных

In [1]:
import warnings; warnings.filterwarnings('ignore')
import os, glob
os.environ.setdefault('LOKY_MAX_CPU_COUNT', '1')
import numpy as np
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from matplotlib.gridspec import GridSpec

from scipy import stats
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.stattools import durbin_watson
from statsmodels.tsa.seasonal import STL

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, RidgeClassifierCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report, roc_auc_score,
    mean_absolute_error, mean_squared_error, roc_curve, auc)
from sklearn.inspection import permutation_importance

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False

# ── TS-классификаторы aeon ───────────────────────────────────────────────────
# При отсутствии пакета в окружении он устанавливается перед импортом.
import subprocess, sys
try:
    import aeon  # noqa: F401
except ImportError:
    print("Устанавливаю aeon ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "aeon"])
    import aeon  # noqa: F401

from aeon.classification.convolution_based import RocketClassifier, MiniRocketClassifier
from aeon.classification.interval_based import (
    TimeSeriesForestClassifier,
    RandomIntervalSpectralEnsembleClassifier,
)

plt.rcParams.update({'figure.dpi': 100, 'font.size': 11, 'axes.titlesize': 13})
PALETTE = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

In [2]:
# ── Загрузка parquet-файлов по годам ─────────────────────────────────────────
CITIES = ['Москва','Санкт-Петербург','Сочи','Геленджик','Благовещенск','Находка']
CLIMATE_MAP = {
    'Москва': 'умеренно-континентальный',
    'Санкт-Петербург': 'умеренно-континентальный',
    'Сочи': 'субтропический',
    'Геленджик': 'субтропический',
    'Благовещенск': 'резко-континентальный',
    'Находка': 'резко-континентальный',
}
LABEL_MAP = {'умеренно-континентальный':0,'субтропический':1,'резко-континентальный':2}
FEATURES = ['temperature_2m','relative_humidity_2m','precipitation',
            'rain','snowfall','weathercode','wind_speed_10m','surface_pressure']

DATA_DIR = Path('data')  # папка с parquet файлами

data = {}
for city in CITIES:
    files = sorted(DATA_DIR.glob(f'{city}_*.parquet'))
    if not files:
        raise FileNotFoundError(f"Файлы для {city} не найдены в {DATA_DIR}")
    dfs = []
    for f in files:
        df = pd.read_parquet(f)
        # Приводим индекс к DatetimeIndex
        if not isinstance(df.index, pd.DatetimeIndex):
            if 'date' in df.columns:
                df = df.set_index('date')
            elif 'time' in df.columns:
                df = df.set_index('time')
        df.index = pd.to_datetime(df.index)
        df = df.sort_index()
        dfs.append(df)
    city_df = pd.concat(dfs).sort_index()
    # Оставляем только нужные колонки (которые есть в файлах)
    cols_present = [c for c in FEATURES if c in city_df.columns]
    data[city] = city_df[cols_present].copy()
    print(f"{city:22s}: {len(city_df)} строк | {city_df.index.min().date()} — {city_df.index.max().date()} | колонки: {cols_present}")

# Обновляем FEATURES под реальные данные
FEATURES = [c for c in FEATURES if all(c in data[city].columns for city in CITIES)]
print(f"\nОбщие признаки: {FEATURES}")

Москва                : 61368 строк | 2019-01-01 — 2025-12-31 | колонки: ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'rain', 'snowfall', 'weathercode', 'wind_speed_10m', 'surface_pressure']
Санкт-Петербург       : 61368 строк | 2019-01-01 — 2025-12-31 | колонки: ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'rain', 'snowfall', 'weathercode', 'wind_speed_10m', 'surface_pressure']
Сочи                  : 61368 строк | 2019-01-01 — 2025-12-31 | колонки: ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'rain', 'snowfall', 'weathercode', 'wind_speed_10m', 'surface_pressure']
Геленджик             : 61368 строк | 2019-01-01 — 2025-12-31 | колонки: ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'rain', 'snowfall', 'weathercode', 'wind_speed_10m', 'surface_pressure']


Благовещенск          : 61368 строк | 2019-01-01 — 2025-12-31 | колонки: ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'rain', 'snowfall', 'weathercode', 'wind_speed_10m', 'surface_pressure']
Находка               : 61368 строк | 2019-01-01 — 2025-12-31 | колонки: ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'rain', 'snowfall', 'weathercode', 'wind_speed_10m', 'surface_pressure']

Общие признаки: ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'rain', 'snowfall', 'weathercode', 'wind_speed_10m', 'surface_pressure']


---
## 2.1. Разведочный анализ данных (EDA)
### 2.1.1. Визуальный анализ временных рядов — тренды, сезонность, выбросы

In [3]:
# Температура по всем городам
fig, axes = plt.subplots(3, 2, figsize=(16, 12))
axes = axes.flatten()
for i, city in enumerate(CITIES):
    df = data[city]
    axes[i].plot(df.index, df['temperature_2m'], lw=0.7, color=PALETTE[i], alpha=0.7)
    roll = df['temperature_2m'].rolling(30, center=True).mean()
    axes[i].plot(df.index, roll, lw=2, color='black', label='MA(30)')
    axes[i].set_title(f"{city} [{CLIMATE_MAP[city]}]")
    axes[i].set_ylabel('°C'); axes[i].legend(fontsize=9); axes[i].grid(alpha=0.3)
    axes[i].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
fig.suptitle('Температура 2m — все города (2019–2025)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [4]:
# Осадки, ветер, давление — обзор
fig, axes = plt.subplots(3, 2, figsize=(16, 12))
axes = axes.flatten()
for i, city in enumerate(CITIES):
    df = data[city]
    ax2 = axes[i].twinx()
    axes[i].bar(df.index, df['precipitation'], color=PALETTE[i], alpha=0.5, width=1, label='Осадки')
    monthly = df['precipitation'].resample('ME').sum()
    ax2.plot(monthly.index, monthly.values, 'k-', lw=1.5, label='Сумма/мес')
    axes[i].set_title(city, fontsize=11)
    axes[i].set_ylabel('мм/день'); ax2.set_ylabel('мм/мес')
    axes[i].grid(alpha=0.2)
fig.suptitle('Суточные осадки и месячные суммы', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [5]:
# Распределения признаков (гистограммы + box-plots по месяцам)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
feat_labels = {'temperature_2m':'Температура (°C)','relative_humidity_2m':'Влажность (%)',
               'precipitation':'Осадки (мм)','rain':'Дождь (мм)',
               'snowfall':'Снег (мм)','weathercode':'Код погоды',
               'wind_speed_10m':'Скорость ветра (м/с)','surface_pressure':'Давление (гПа)'}
for j, feat in enumerate(FEATURES[:8]):
    ax = axes[j//4, j%4]
    for i, city in enumerate(CITIES):
        if feat in data[city].columns:
            ax.hist(data[city][feat].dropna(), bins=40, alpha=0.4,
                    color=PALETTE[i], label=city, density=True)
    ax.set_title(feat_labels.get(feat, feat), fontsize=10); ax.grid(alpha=0.3)
    if j == 0: ax.legend(fontsize=7)
fig.suptitle('Распределение метеопризнаков по городам', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [6]:
# Климатические профили — box-plot температуры по месяцам
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
for i, city in enumerate(CITIES):
    df = data[city].copy(); df['month'] = df.index.month
    mdata = [df[df['month']==m]['temperature_2m'].dropna().values for m in range(1,13)]
    bp = axes[i].boxplot(mdata, patch_artist=True, medianprops=dict(color='black',linewidth=2))
    for patch in bp['boxes']: patch.set_facecolor(PALETTE[i]); patch.set_alpha(0.7)
    axes[i].set_title(city, fontsize=11); axes[i].set_ylabel('°C')
    axes[i].set_xticklabels(['Янв','Фев','Мар','Апр','Май','Июн',
                              'Июл','Авг','Сен','Окт','Ноя','Дек'], fontsize=8)
    axes[i].axhline(0, color='red', lw=0.8, ls='--', alpha=0.7); axes[i].grid(alpha=0.3)
fig.suptitle('Температура по месяцам (box-plot)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [7]:
# Q-Q plots нормальности температуры
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for i, city in enumerate(CITIES):
    stats.probplot(data[city]['temperature_2m'].dropna(), dist='norm', plot=axes[i])
    axes[i].set_title(f'Q-Q: {city}', fontsize=10); axes[i].grid(alpha=0.3)
fig.suptitle('Q-Q plots нормальности температуры', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

### 2.1.2. Кросс-городской анализ

In [8]:
# Сравнение климатических профилей
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
month_names = ['Янв','Фев','Мар','Апр','Май','Июн','Июл','Авг','Сен','Окт','Ноя','Дек']
for i, city in enumerate(CITIES):
    df = data[city].copy(); df['month'] = df.index.month
    axes[0].plot(range(1,13), df.groupby('month')['temperature_2m'].mean().values,
                 '-o', color=PALETTE[i], label=city, lw=2, markersize=5)
axes[0].set_title('Среднемесячная температура', fontsize=12)
axes[0].set_xticks(range(1,13)); axes[0].set_xticklabels(month_names, fontsize=9)
axes[0].axhline(0,color='gray',lw=0.8,ls='--'); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

annual = [data[c]['precipitation'].resample('YE').sum().mean() for c in CITIES]
bars = axes[1].bar(CITIES, annual, color=PALETTE, alpha=0.8)
axes[1].set_title('Среднегодовые осадки (мм)', fontsize=12)
axes[1].set_xticklabels(CITIES, rotation=20, ha='right', fontsize=9)
for bar, val in zip(bars, annual):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+5, f'{val:.0f}', ha='center', fontsize=9)
axes[1].grid(alpha=0.3, axis='y')
fig.suptitle('Климатические профили городов', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [9]:
# Матрицы корреляций
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, city in enumerate(CITIES):
    corr = data[city][FEATURES].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, ax=axes[i], mask=mask, annot=True, fmt='.2f',
                cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                annot_kws={'size':7}, linewidths=0.5)
    axes[i].set_title(f'Корреляции: {city}', fontsize=10)
    axes[i].tick_params(labelsize=7)
fig.suptitle('Корреляционные матрицы метеопризнаков', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [10]:
# Анализ пропусков
print('=== Анализ пропусков ===')
for city in CITIES:
    mis = data[city].isnull().sum()
    total = len(data[city])
    if mis.sum() == 0:
        print(f'{city:22s}: пропусков нет')
    else:
        pct = (mis[mis>0] / total * 100).round(2)
        print(f'{city:22s}: {pct.to_dict()}')
        # Заполняем линейной интерполяцией
        data[city] = data[city].interpolate(method='time').ffill().bfill()
        print(f'  -> интерполяция применена')
print('\nСтратегия: линейная интерполяция для коротких пропусков (<3 дней).')
print('Выбросы сохраняются — экстремальные явления могут быть реальными.')

=== Анализ пропусков ===
Москва                : пропусков нет
Санкт-Петербург       : пропусков нет
Сочи                  : пропусков нет
Геленджик             : пропусков нет
Благовещенск          : пропусков нет
Находка               : пропусков нет

Стратегия: линейная интерполяция для коротких пропусков (<3 дней).
Выбросы сохраняются — экстремальные явления могут быть реальными.


### 2.1.3. Анализ стационарности

In [11]:
# Тест Дики-Фуллера
print('=== Расширенный тест Дики-Фуллера (ADF) ===')
print('{:<22s} {:>14s} {:>10s} {:>8s}'.format('Город','ADF-статистика','p-value','Стац.'))
print('-'*58)
for city in CITIES:
    temp = data[city]['temperature_2m'].dropna()
    adf_stat, p_val = adfuller(temp, autolag='AIC')[:2]
    mark = 'ДА' if p_val < 0.05 else 'НЕТ'
    print('{:<22s} {:>14.4f} {:>10.4f} {:>8s}'.format(city, adf_stat, p_val, mark))
print('\nВывод: температурные ряды стационарны (ADF p<0.05) — сезонная периодичность')
print('уже присутствует в данных. Дифференцирование не требуется для классификации.')

=== Расширенный тест Дики-Фуллера (ADF) ===
Город                  ADF-статистика    p-value    Стац.
----------------------------------------------------------


Москва                        -6.2192     0.0000       ДА


Санкт-Петербург               -6.0422     0.0000       ДА


Сочи                          -6.4076     0.0000       ДА


Геленджик                     -6.8592     0.0000       ДА


Благовещенск                  -3.8911     0.0021       ДА


Находка                       -4.9382     0.0000       ДА

Вывод: температурные ряды стационарны (ADF p<0.05) — сезонная периодичность
уже присутствует в данных. Дифференцирование не требуется для классификации.


In [12]:
# ACF/PACF + STL разложение (пример — Москва)
city_ex = 'Москва'
temp_ex = data[city_ex]['temperature_2m'].dropna()
nlags = 60

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
acf_v = acf(temp_ex.values[:730], nlags=nlags, fft=True)
pacf_v = pacf(temp_ex.values[:730], nlags=nlags, method='ywm')
ci = 1.96/np.sqrt(730)

axes[0,0].stem(range(nlags+1), acf_v, linefmt='b-', markerfmt='bo', basefmt='k-')
axes[0,0].axhline(ci,ls='--',color='red',alpha=0.7); axes[0,0].axhline(-ci,ls='--',color='red',alpha=0.7)
axes[0,0].set_title(f'ACF температуры ({city_ex})'); axes[0,0].grid(alpha=0.3)

axes[0,1].stem(range(nlags+1), pacf_v, linefmt='g-', markerfmt='go', basefmt='k-')
axes[0,1].axhline(ci,ls='--',color='red',alpha=0.7); axes[0,1].axhline(-ci,ls='--',color='red',alpha=0.7)
axes[0,1].set_title(f'PACF температуры ({city_ex})'); axes[0,1].grid(alpha=0.3)

stl = STL(temp_ex, period=365, robust=True).fit()
axes[1,0].plot(temp_ex.index, stl.trend, color='blue', lw=1.5)
axes[1,0].set_title('Тренд STL'); axes[1,0].set_ylabel('°C'); axes[1,0].grid(alpha=0.3)
axes[1,1].plot(temp_ex.index, stl.seasonal, color='orange', lw=0.8)
axes[1,1].set_title('Сезонность STL'); axes[1,1].set_ylabel('°C'); axes[1,1].grid(alpha=0.3)

fig.suptitle(f'ACF/PACF и STL-разложение — {city_ex}', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

### Аналитические выводы по разделу 2.1

1. **Температура** во всех городах имеет выраженную годовую сезонность. Максимальная сезонная амплитуда наблюдается у Благовещенска, минимальная — у Сочи и Геленджика, что соответствует различию между резко-континентальным и субтропическим типами климата.
2. **Осадки и влажность** дополняют температурный профиль: южные города отличаются более мягкой зимой, а города Дальнего Востока — более выраженными сезонными контрастами.
3. **Пропуски** в исходных данных не обнаружены: по всем шести городам загружено по 61368 строк за период 2019-01-01 — 2025-12-31.
4. **ADF-тест** для температуры дал p-value < 0.05 во всех городах (например, Москва: ADF=-6.2192, Благовещенск: ADF=-3.8911), поэтому ряды можно считать стационарными в рамках выбранной проверки.
5. **Корреляционные матрицы и STL-разложение** подтверждают наличие устойчивой сезонной компоненты, которую нужно учитывать и при классификации климата, и при прогнозировании температуры.


---
## 2.2. Инжиниринг признаков для временных рядов

In [13]:
def build_weather_features(df, city, lag_days=[1,2,3,7,14,30], windows=[7,14,30,90]):
    """
    Строит признаки для прогнозирования и агрегированные признаки для классификации.
    lag_days : 1-3 — краткосрочная память атмосферы; 7/14 — синоптические циклы; 30 — месячный
    windows  : 7 — синоптический; 14 — декадный; 30 — месячный; 90 — сезонный
    """
    feat = df.copy()

    # Временные признаки
    feat['day_of_year'] = feat.index.dayofyear
    feat['month']       = feat.index.month
    feat['season']      = feat.index.month % 12 // 3
    feat['year']        = feat.index.year

    # Циклические (sin/cos)
    feat['month_sin'] = np.sin(2*np.pi*feat['month']/12)
    feat['month_cos'] = np.cos(2*np.pi*feat['month']/12)
    feat['doy_sin']   = np.sin(2*np.pi*feat['day_of_year']/365.25)
    feat['doy_cos']   = np.cos(2*np.pi*feat['day_of_year']/365.25)

    # Лаговые признаки
    for col in ['temperature_2m','precipitation','surface_pressure','wind_speed_10m']:
        if col in df.columns:
            for lag in lag_days:
                feat[f'{col}_lag{lag}'] = feat[col].shift(lag)

    # Скользящие статистики
    for col in ['temperature_2m','precipitation','wind_speed_10m','surface_pressure']:
        if col in df.columns:
            for w in windows:
                feat[f'{col}_rmean{w}'] = feat[col].rolling(w, min_periods=1).mean()
                feat[f'{col}_rstd{w}']  = feat[col].rolling(w, min_periods=1).std()
                if col in ['temperature_2m','precipitation']:
                    feat[f'{col}_rmin{w}'] = feat[col].rolling(w, min_periods=1).min()
                    feat[f'{col}_rmax{w}'] = feat[col].rolling(w, min_periods=1).max()

    # Производные
    feat['temp_diff1'] = feat['temperature_2m'].diff(1)
    feat['temp_diff7'] = feat['temperature_2m'].diff(7)
    feat['pressure_diff1'] = feat['surface_pressure'].diff(1) if 'surface_pressure' in feat.columns else 0

    # Амплитуда и дни с осадками
    feat['temp_amplitude_30d'] = feat['temperature_2m'].rolling(30).max() - feat['temperature_2m'].rolling(30).min()
    if 'precipitation' in feat.columns:
        feat['rainy_days_30d'] = (feat['precipitation'] > 0.5).astype(int).rolling(30).sum()

    # Климатическая норма
    doy_mean = feat.groupby(feat.index.dayofyear)['temperature_2m'].transform('mean')
    feat['temp_climate_norm'] = doy_mean
    feat['temp_anomaly'] = feat['temperature_2m'] - doy_mean

    feat['city'] = city
    feat['climate_label'] = CLIMATE_MAP[city]
    feat['climate_id'] = LABEL_MAP[CLIMATE_MAP[city]]
    return feat.dropna()

print('Создание признаков...')
features_data = {}
for city in CITIES:
    features_data[city] = build_weather_features(data[city], city)
    n = len([c for c in features_data[city].columns if c not in ['city','climate_label','climate_id']])
    print(f'{city:22s}: {len(features_data[city])} строк, {n} признаков')
print('Готово!')

Создание признаков...
Москва                : 61338 строк, 95 признаков


Санкт-Петербург       : 61338 строк, 95 признаков
Сочи                  : 61338 строк, 95 признаков


Геленджик             : 61338 строк, 95 признаков
Благовещенск          : 61338 строк, 95 признаков


Находка               : 61338 строк, 95 признаков
Готово!


In [14]:
# PCA — разделимость климатических типов
all_feat = pd.concat(features_data.values())
numeric_cols = [c for c in all_feat.select_dtypes(include=[np.number]).columns
                if c not in ['climate_id','season']]

scaler_pca = StandardScaler()
X_pca_s = scaler_pca.fit_transform(all_feat[numeric_cols].values)
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_pca_s)

print(f'PCA объяснённая дисперсия: PC1={pca.explained_variance_ratio_[0]:.1%}, '
      f'PC2={pca.explained_variance_ratio_[1]:.1%}, суммарно={sum(pca.explained_variance_ratio_[:2]):.1%}')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
climate_colors = {'умеренно-континентальный':'#1f77b4','субтропический':'#ff7f0e','резко-континентальный':'#2ca02c'}
for climate, color in climate_colors.items():
    mask = all_feat['climate_label'] == climate
    axes[0].scatter(X_pca[mask,0], X_pca[mask,1], c=color, alpha=0.3, s=5, label=climate)
axes[0].set_title('PCA: по типу климата'); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')

for i, city in enumerate(CITIES):
    mask = all_feat['city'] == city
    axes[1].scatter(X_pca[mask,0], X_pca[mask,1], c=PALETTE[i], alpha=0.4, s=5, label=city)
axes[1].set_title('PCA: по городам'); axes[1].legend(fontsize=9, markerscale=3); axes[1].grid(alpha=0.3)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')

fig.suptitle('Разделимость городов в пространстве PCA-признаков', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

PCA объяснённая дисперсия: PC1=23.7%, PC2=14.8%, суммарно=38.5%


### Аналитические выводы по разделу 2.2

1. **Сформировано 95 признаков** для каждого города; после лагов и скользящих окон осталось по 61338 наблюдений.
2. **Циклические признаки** месяца и дня года корректно описывают годовую сезонность без искусственного разрыва между декабрём и январём.
3. **Лаги и скользящие статистики** отражают краткосрочную инерцию погоды, недельные изменения и более медленный сезонный фон.
4. **Климатическая норма** по дню года используется как ориентир ожидаемой температуры и особенно важна для прогноза на горизонт 30 дней.
5. **PCA** по числовым признакам объясняет 38.5% дисперсии первыми двумя компонентами (PC1=23.7%, PC2=14.8%). Этого достаточно для визуального разделения части климатических режимов, но не заменяет специализированную модель временных рядов.


---
## 2.3. Построение модели классификации

### Обоснование архитектуры

**Задача:** по фрагменту временного ряда метеопараметров определить тип климата.

**Почему нужны специализированные TS-классификаторы:**
Классические ML-модели (RF, GBM) на агрегированных признаках теряют информацию о порядке точек и локальных паттернах. Специализированные методы работают напрямую с сырым рядом.

| Модель | Принцип | Преимущество для метеоданных |
|--------|---------|------------------------------|
| **ROCKET** | Случайные свёрточные ядра (PPV + Max) | SOTA-качество, быстро |
| **MiniRocket** | Детерминированные ядра, в 75× быстрее | Воспроизводимость |
| **Time Series Forest** | Деревья на случайных интервалах | Интерпретируемость |
| **RISE** | Спектральные признаки из интервалов | Эффективен для сезонных данных |

Эти 4 модели обучаются ниже **безусловно** и сравниваются между собой — выбор лучшей делается по тесту 2025.

**Не подходят:**
- *Логистическая регрессия на сырых рядах* — предполагает независимость точек (нарушается автокорреляцией)
- *k-NN с евклидовым расстоянием* — игнорирует временные сдвиги и деформации
- *Наивный Байес* — предполагает независимость каждого измерения во времени

**Длина окна для классификации: 30 последовательных наблюдений**

In [15]:
# ── Подготовка 3D-тензора (n_samples, n_channels, timesteps) ─────────────────
W_CLS = 30
TS_FEATURES = [f for f in ['temperature_2m','relative_humidity_2m','precipitation',
                             'wind_speed_10m','surface_pressure'] if f in FEATURES]

def build_ts_dataset(data_dict, window=W_CLS, step=7):
    """
    Формат: (n_samples, n_channels, window)
    step=7: новый пример каждые 7 наблюдений — баланс между объёмом и независимостью окон
    """
    X_list, y_list, meta_list = [], [], []
    for city in CITIES:
        df = data_dict[city][TS_FEATURES].copy()
        df_norm = (df - df.mean()) / (df.std() + 1e-8)
        arr = df_norm.values
        climate_id = LABEL_MAP[CLIMATE_MAP[city]]
        for start in range(0, len(arr)-window, step):
            X_list.append(arr[start:start+window].T)
            y_list.append(climate_id)
            meta_list.append((city, df.index[start+window-1]))
    return np.array(X_list, dtype=np.float32), np.array(y_list), meta_list

X_ts, y_ts, meta_ts = build_ts_dataset(data, window=W_CLS, step=7)
years_ts = np.array([m[1].year for m in meta_ts])

train_m = years_ts <= 2023
val_m   = years_ts == 2024
test_m  = years_ts == 2025

X_tr, y_tr = X_ts[train_m], y_ts[train_m]
X_vl, y_vl = X_ts[val_m],   y_ts[val_m]
X_te, y_te = X_ts[test_m],  y_ts[test_m]

print(f'X shape: {X_ts.shape}  (samples × channels × timesteps)')
print(f'Каналы ({X_ts.shape[1]}): {TS_FEATURES}')
print(f'Train: {len(y_tr)} | Val: {len(y_vl)} | Test: {len(y_te)}')
print('Классы:', {0:'умеренно-конт.',1:'субтропич.',2:'резко-конт.'})

X shape: (52578, 5, 30)  (samples × channels × timesteps)
Каналы (5): ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m', 'surface_pressure']
Train: 37542 | Val: 7530 | Test: 7506
Классы: {0: 'умеренно-конт.', 1: 'субтропич.', 2: 'резко-конт.'}


In [16]:
# ── Обучение TS-классификаторов ───────────────────────────────────────────────
# Сравниваются четыре специализированные модели классификации временных рядов.
import time
ts_models = {}
ts_val_acc = {}

def _fit_and_report(name, model, X_train, y_train, X_val, y_val):
    t0 = time.time()
    model.fit(X_train, y_train)
    fit_s = time.time() - t0
    val_acc = accuracy_score(y_val, model.predict(X_val))
    ts_models[name] = model
    ts_val_acc[name] = val_acc
    print(f"  {name:<12s} Val Acc = {val_acc:.4f}   (fit {fit_s:5.1f} c)")

print("Обучение ROCKET...")
_fit_and_report(
    'ROCKET',
    RocketClassifier(n_kernels=2000, n_jobs=-1, random_state=42),
    X_tr, y_tr, X_vl, y_vl,
)

print("Обучение MiniRocket...")
_fit_and_report(
    'MiniRocket',
    MiniRocketClassifier(n_jobs=-1, random_state=42),
    X_tr, y_tr, X_vl, y_vl,
)

print("Обучение TimeSeriesForest...")
_fit_and_report(
    'TSForest',
    TimeSeriesForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    X_tr, y_tr, X_vl, y_vl,
)

print("Обучение RISE (Random Interval Spectral Ensemble)...")
_fit_and_report(
    'RISE',
    RandomIntervalSpectralEnsembleClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    X_tr, y_tr, X_vl, y_vl,
)

print(f"\nОбучено моделей: {list(ts_models.keys())}")
print("Val Accuracy:", {k: round(v, 4) for k, v in ts_val_acc.items()})

Обучение ROCKET...


  ROCKET       Val Acc = 0.8179   (fit  47.3 c)
Обучение MiniRocket...


  MiniRocket   Val Acc = 1.0000   (fit 347.0 c)
Обучение TimeSeriesForest...


  TSForest     Val Acc = 0.9992   (fit 660.9 c)
Обучение RISE (Random Interval Spectral Ensemble)...


  RISE         Val Acc = 1.0000   (fit  26.2 c)

Обучено моделей: ['ROCKET', 'MiniRocket', 'TSForest', 'RISE']
Val Accuracy: {'ROCKET': 0.8179, 'MiniRocket': 1.0, 'TSForest': 0.9992, 'RISE': 1.0}


## 2.4. Оценка качества классификации

In [17]:
# ── Метрики TS-классификаторов на тесте 2025 ─────────────────────────────────
print('='*80)
print('{:<22s} {:>7s} {:>8s} {:>8s} {:>9s}'.format('Модель','Acc','F1-mac','F1-wt','ROC-AUC'))
print('='*80)

ts_test_res = {}
best_name, best_acc = None, -1.0

for name, model in ts_models.items():
    y_pred = model.predict(X_te)

    # Для моделей без predict_proba ROC-AUC не рассчитывается.
    try:
        y_prob = model.predict_proba(X_te)
    except (AttributeError, NotImplementedError):
        y_prob = None

    acc = accuracy_score(y_te, y_pred)
    f1m = f1_score(y_te, y_pred, average='macro',    zero_division=0)
    f1w = f1_score(y_te, y_pred, average='weighted', zero_division=0)

    if y_prob is not None and y_prob.shape[1] == 3:
        try:
            roc = roc_auc_score(y_te, y_prob, multi_class='ovr', average='macro')
        except ValueError:
            roc = float('nan')
    else:
        roc = float('nan')

    mark = ' ◀ ЛУЧШАЯ' if acc > best_acc else ''
    print('{:<22s} {:>7.4f} {:>8.4f} {:>8.4f} {:>9.4f}{}'.format(name, acc, f1m, f1w, roc, mark))
    ts_test_res[name] = dict(y_pred=y_pred, y_prob=y_prob, acc=acc, f1m=f1m, f1w=f1w, roc=roc)
    if acc > best_acc:
        best_acc, best_name = acc, name

print('='*80)
best_clf_ts = ts_models[best_name]
print(f'\nЛучшая модель на тесте 2025: {best_name}  (Accuracy = {best_acc:.4f})')


Модель                     Acc   F1-mac    F1-wt   ROC-AUC


ROCKET                  0.7823   0.7822   0.7822    0.8367 ◀ ЛУЧШАЯ


MiniRocket              1.0000   1.0000   1.0000    1.0000 ◀ ЛУЧШАЯ


TSForest                0.9977   0.9977   0.9977    1.0000


RISE                    1.0000   1.0000   1.0000    1.0000

Лучшая модель на тесте 2025: MiniRocket  (Accuracy = 1.0000)


In [18]:
# Confusion matrix + ROC
cls_labels = ['Умеренно-конт.','Субтропич.','Резко-конт.']
y_pred_b = ts_test_res[best_name]['y_pred']
y_prob_b = ts_test_res[best_name]['y_prob']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Абсолютная CM
cm = confusion_matrix(y_te, y_pred_b)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=cls_labels, yticklabels=cls_labels, linewidths=0.5)
axes[0].set_title(f'CM: {best_name}'); axes[0].set_xlabel('Предсказан'); axes[0].set_ylabel('Истина')

# Нормализованная CM
cm_n = cm.astype(float)/cm.sum(axis=1,keepdims=True)
sns.heatmap(cm_n, annot=True, fmt='.2f', cmap='Blues', ax=axes[1],
            xticklabels=cls_labels, yticklabels=cls_labels, linewidths=0.5)
axes[1].set_title('CM нормализованная'); axes[1].set_xlabel('Предсказан'); axes[1].set_ylabel('Истина')

# ROC
if y_prob_b is not None and y_prob_b.shape[1]==3:
    y_bin = label_binarize(y_te, classes=[0,1,2])
    for i,(lbl,col) in enumerate(zip(cls_labels,['blue','orange','green'])):
        fpr,tpr,_ = roc_curve(y_bin[:,i], y_prob_b[:,i])
        axes[2].plot(fpr,tpr,color=col,lw=2,label=f'{lbl} (AUC={auc(fpr,tpr):.3f})')
    axes[2].plot([0,1],[0,1],'k--',lw=1); axes[2].set_xlabel('FPR'); axes[2].set_ylabel('TPR')
    axes[2].set_title('ROC-кривые (OvR)'); axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3)
else:
    axes[2].text(0.5,0.5,'predict_proba недоступен',ha='center',va='center',transform=axes[2].transAxes)

fig.suptitle(f'Оценка классификатора {best_name}', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print(classification_report(y_te, y_pred_b, target_names=cls_labels))

                precision    recall  f1-score   support

Умеренно-конт.       1.00      1.00      1.00      2502
    Субтропич.       1.00      1.00      1.00      2502
   Резко-конт.       1.00      1.00      1.00      2502

      accuracy                           1.00      7506
     macro avg       1.00      1.00      1.00      7506
  weighted avg       1.00      1.00      1.00      7506



In [19]:
# Точность по месяцам + важность каналов
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

meta_te = [m for m,f in zip(meta_ts,test_m) if f]
months_te = np.array([m[1].month for m in meta_te])
month_labels = ['Янв','Фев','Мар','Апр','Май','Июн','Июл','Авг','Сен','Окт','Ноя','Дек']

monthly_acc = {}
for mo in range(1,13):
    mk = months_te==mo
    if mk.sum()>0:
        monthly_acc[mo] = (y_pred_b[mk]==y_te[mk]).mean()*100

axes[0].bar(range(1,13),[monthly_acc.get(m,0) for m in range(1,13)],
            color=['#1f77b4' if monthly_acc.get(m,0)>=90 else '#ff7f0e' for m in range(1,13)],alpha=0.8)
axes[0].axhline(90,color='red',ls='--',lw=1,alpha=0.7,label='90%')
axes[0].set_title(f'Accuracy по месяцам: {best_name}')
axes[0].set_xticks(range(1,13)); axes[0].set_xticklabels(month_labels,fontsize=9)
axes[0].set_ylim(0,108); axes[0].legend(); axes[0].grid(alpha=0.3,axis='y')

# Permutation importance по каналам
base_acc = accuracy_score(y_te, y_pred_b)
ch_imp = []
for ch in range(len(TS_FEATURES)):
    Xp = X_te.copy()
    Xp[:,ch,:] = Xp[np.random.permutation(len(Xp)),ch,:]
    ch_imp.append(base_acc - accuracy_score(y_te, best_clf_ts.predict(Xp)))

colors_ch = ['#d62728' if v>0 else '#1f77b4' for v in ch_imp]
axes[1].barh(TS_FEATURES, ch_imp, color=colors_ch, alpha=0.85)
axes[1].axvline(0,color='black',lw=1)
axes[1].set_title(f'Важность каналов (permutation): {best_name}')
axes[1].set_xlabel('Снижение Accuracy'); axes[1].grid(alpha=0.3,axis='x')

plt.tight_layout(); plt.show()

### Аналитические выводы по разделам 2.3–2.4

1. Для классификации построен тензор размера **52578 × 5 × 30**: 52578 окон, 5 метеоканалов и 30 последовательных наблюдений в каждом окне.
2. На валидации лучшими оказались **MiniRocket** и **RISE** с Accuracy=1.0000; **TSForest** также показал почти идеальный результат (0.9992), а **ROCKET** заметно уступил (0.8179).
3. На тесте 2025 года лучшей моделью выбран **MiniRocket**: Accuracy=1.0000, F1-macro=1.0000, F1-weighted=1.0000, ROC-AUC=1.0000.
4. Confusion matrix показывает отсутствие ошибок на тестовой выборке: каждый из трёх климатических классов имеет support=2502 и классифицируется с precision/recall/F1=1.00.
5. Полученный результат объясняется тем, что классы заданы по городам с устойчиво различающимися климатическими профилями; окна из температуры, влажности, осадков, ветра и давления достаточно для разделения этих групп.


---
## 2.5. Построение моделей прогнозирования

### Обоснование

**Горизонт H=30 дней.** Разбивка: train 2019–2023, val 2024, test 2025.

Разные модели для разных климатических зон:
- **Субтропический** (Сочи, Геленджик): Ridge Regression — температура устойчива, мало дисперсии
- **Умеренно-континентальный** (Москва, СПб): Random Forest — нелинейные переходы между сезонами
- **Резко-континентальный** (Благовещенск, Находка): Gradient Boosting / LightGBM — большая амплитуда, сложные нелинейности

**Не подходят для горизонта 30 дней:**
- Наивный прогноз (последнее значение) — ошибка ~амплитуда сезона
- ARIMA без сезонной компоненты — не учитывает годовую сезонность

In [3]:
def build_forecast_features(df, horizon=30, n_lags=60, windows=[7,14,30], drop_target_na=True):
    feat = pd.DataFrame(index=df.index)
    feat['doy_sin'] = np.sin(2*np.pi*df.index.dayofyear/365.25)
    feat['doy_cos'] = np.cos(2*np.pi*df.index.dayofyear/365.25)
    feat['month_sin'] = np.sin(2*np.pi*df.index.month/12)
    feat['month_cos'] = np.cos(2*np.pi*df.index.month/12)
    feat['year_norm'] = (df.index.year - 2019) / 6.0

    for lag in range(1, n_lags+1):
        feat[f'temp_lag{lag}'] = df['temperature_2m'].shift(lag)

    for w in windows:
        feat[f'temp_rmean{w}'] = df['temperature_2m'].rolling(w).mean()
        feat[f'temp_rstd{w}']  = df['temperature_2m'].rolling(w).std()
        feat[f'temp_rmin{w}']  = df['temperature_2m'].rolling(w).min()
        feat[f'temp_rmax{w}']  = df['temperature_2m'].rolling(w).max()

    feat['temp_diff1']  = df['temperature_2m'].diff(1)
    feat['temp_diff7']  = df['temperature_2m'].diff(7)
    feat['temp_diff30'] = df['temperature_2m'].diff(30)

    # Климатическая норма целевого дня
    tgt_doy = (df.index.dayofyear + horizon - 1) % 365 + 1
    doy_mean = df.groupby(df.index.dayofyear)['temperature_2m'].mean()
    feat['target_climate_norm'] = [doy_mean.get(d, doy_mean.mean()) for d in tgt_doy]

    for col in [c for c in ['relative_humidity_2m','surface_pressure','wind_speed_10m'] if c in df.columns]:
        feat[f'{col}_lag1']     = df[col].shift(1)
        feat[f'{col}_rmean14']  = df[col].rolling(14).mean()

    feat['target'] = df['temperature_2m'].shift(-horizon)
    if drop_target_na:
        return feat.dropna()
    feature_cols = [c for c in feat.columns if c != 'target']
    return feat.dropna(subset=feature_cols)

MODEL_CFG = {
    'Москва':'rf','Санкт-Петербург':'rf',
    'Сочи':'ridge','Геленджик':'ridge',
    'Благовещенск':'gb','Находка':'gb'
}
HORIZON_DAYS = 30
DATA_STEP = data[CITIES[0]].index.to_series().diff().dropna().median()
STEPS_PER_DAY = int(pd.Timedelta(days=1) / DATA_STEP)
HORIZON = HORIZON_DAYS * STEPS_PER_DAY

forecast_models, forecast_feat, forecast_feat_full = {}, {}, {}
print(f'Горизонт прогноза: {HORIZON_DAYS} дней ({HORIZON} временных шагов)')
for city in CITIES:
    fd_full = build_forecast_features(data[city], horizon=HORIZON, drop_target_na=False)
    fd = fd_full.dropna(subset=['target'])
    yr = fd.index.year
    X, y = fd.drop('target',axis=1).values, fd['target'].values
    X_tr2 = X[yr<=2023]; y_tr2 = y[yr<=2023]
    X_vl2 = X[yr==2024]; y_vl2 = y[yr==2024]
    sc = StandardScaler(); X_tr2s = sc.fit_transform(X_tr2); X_vl2s = sc.transform(X_vl2)

    mt = MODEL_CFG[city]
    if mt=='rf':    m = RandomForestRegressor(n_estimators=200,max_depth=12,min_samples_leaf=5,random_state=42,n_jobs=1)
    elif mt=='ridge': m = Ridge(alpha=10.0)
    else: m = (lgb.LGBMRegressor(n_estimators=300,learning_rate=0.05,num_leaves=31,random_state=42,verbose=-1)
               if HAS_LGB else GradientBoostingRegressor(n_estimators=200,learning_rate=0.05,max_depth=5,random_state=42))
    m.fit(X_tr2s, y_tr2)
    mae_v = mean_absolute_error(y_vl2, m.predict(X_vl2s))
    print(f'{city:22s} [{mt}] VAL MAE={mae_v:.2f}°C')
    forecast_models[city] = {'model':m,'scaler':sc}
    forecast_feat[city] = fd
    forecast_feat_full[city] = fd_full

Горизонт прогноза: 30 дней (720 временных шагов)


Москва                 [rf] VAL MAE=4.41°C


Санкт-Петербург        [rf] VAL MAE=4.19°C


Сочи                   [ridge] VAL MAE=2.67°C


Геленджик              [ridge] VAL MAE=3.26°C


Благовещенск           [gb] VAL MAE=3.47°C


Находка                [gb] VAL MAE=2.88°C


## 2.6. Оценка качества прогнозирования

In [4]:
def directional_accuracy(yt, yp):
    return np.mean(np.sign(np.diff(yt))==np.sign(np.diff(yp)))*100

print('='*82)
print('MAPE* считается только для дней с |Tфакт| >= 1°C; для температуры в °C основная относительная метрика — WAPE.')
print('{:<22s} {:>7s} {:>7s} {:>7s} {:>7s} {:>7s} {:>8s}'.format(
    'Город','MAE','RMSE','MAPE*%','WAPE%','R2','DA%'))
print('='*82)

all_test_res = {}
for city in CITIES:
    fd = forecast_feat[city]; yr = fd.index.year
    X = fd.drop('target',axis=1).values; y = fd['target'].values
    mask = yr==2025
    Xte = forecast_models[city]['scaler'].transform(X[mask])
    yte = y[mask]; yp = forecast_models[city]['model'].predict(Xte)
    mae  = mean_absolute_error(yte,yp)
    rmse = np.sqrt(mean_squared_error(yte,yp))
    mape_mask = np.abs(yte) >= 1.0
    mape = np.mean(np.abs((yte[mape_mask]-yp[mape_mask])/np.abs(yte[mape_mask])))*100
    wape = np.sum(np.abs(yte-yp))/np.sum(np.abs(yte))*100
    r2   = 1 - np.sum((yte-yp)**2)/np.sum((yte-yte.mean())**2)
    da   = directional_accuracy(yte,yp)
    all_test_res[city] = dict(y_true=yte,y_pred=yp,idx=fd.index[mask],
                               mae=mae,rmse=rmse,mape=mape,wape=wape,r2=r2,da=da)
    print('{:<22s} {:>7.2f} {:>7.2f} {:>7.1f} {:>7.1f} {:>7.4f} {:>8.1f}'.format(
        city,mae,rmse,mape,wape,r2,da))
print('='*82)

MAPE* считается только для дней с |Tфакт| >= 1°C; для температуры в °C основная относительная метрика — WAPE.
Город                      MAE    RMSE  MAPE*%   WAPE%      R2      DA%
Москва                    4.53    5.52    75.7    43.5  0.6555     64.8
Санкт-Петербург           4.15    5.25    86.5    43.7  0.6246     62.5


Сочи                      2.92    3.78    33.9    19.1  0.7602     71.0


Геленджик                 3.27    4.14    40.3    21.9  0.7734     71.7
Благовещенск              3.44    4.60    44.4    24.3  0.9151     71.7
Находка                   2.76    3.68    48.9    22.7  0.9016     71.5


In [22]:
# Прогноз vs факт
fig, axes = plt.subplots(3, 2, figsize=(16, 13))
axes = axes.flatten()
for i, city in enumerate(CITIES):
    res = all_test_res[city]
    axes[i].plot(res['idx'], res['y_true'],'b-',lw=1.5,label='Факт',alpha=0.9)
    axes[i].plot(res['idx'], res['y_pred'],'r--',lw=1.5,label=f'Прогноз +{HORIZON}д',alpha=0.9)
    axes[i].fill_between(res['idx'], res['y_pred']-res['rmse'], res['y_pred']+res['rmse'],
                          alpha=0.2, color='red', label=f'±RMSE={res["rmse"]:.1f}°C')
    axes[i].set_title(f"{city} | MAE={res['mae']:.1f}°C R²={res['r2']:.3f}")
    axes[i].set_ylabel('°C'); axes[i].legend(fontsize=8); axes[i].grid(alpha=0.3)
    axes[i].xaxis.set_major_formatter(mdates.DateFormatter('%m/%Y'))
fig.suptitle('Прогноз температуры на 30 дней вперёд — тест 2025',fontsize=13,fontweight='bold')
plt.tight_layout(); plt.show()

In [23]:
# Box-plot ошибок
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
errs = {c: all_test_res[c]['y_true']-all_test_res[c]['y_pred'] for c in CITIES}
bp = axes[0].boxplot(errs.values(), labels=CITIES, patch_artist=True,
                     medianprops=dict(color='black',linewidth=2))
for patch,col in zip(bp['boxes'],PALETTE): patch.set_facecolor(col); patch.set_alpha(0.7)
axes[0].axhline(0,color='black',lw=1,ls='--')
axes[0].set_title('Распределение ошибок прогноза'); axes[0].set_ylabel('Ошибка (°C)')
axes[0].set_xticklabels(CITIES,rotation=20,ha='right',fontsize=9); axes[0].grid(alpha=0.3,axis='y')

# Сводная тепловая карта метрик
vals = np.array([[all_test_res[c][k] for k in ['mae','rmse','mape','wape','r2']] for c in CITIES])
im = axes[1].imshow(vals.T, cmap='YlOrRd', aspect='auto')
axes[1].set_xticks(range(len(CITIES))); axes[1].set_xticklabels(CITIES,rotation=20,ha='right',fontsize=9)
axes[1].set_yticks(range(5)); axes[1].set_yticklabels(['MAE','RMSE','MAPE%','WAPE%','R²'])
axes[1].set_title('Метрики прогнозирования')
for i in range(len(CITIES)):
    for j in range(5): axes[1].text(i,j,f'{vals[i,j]:.2f}',ha='center',va='center',fontsize=8)
plt.colorbar(im,ax=axes[1])
plt.tight_layout(); plt.show()

In [5]:
# Анализ остатков
print('='*75)
print('{:<22s} {:>10s} {:>10s} {:>10s} {:>8s} {:>10s}'.format(
    'Город','SW p-val','JB p-val','LB p-val','DW','BP p-val'))
print('-'*75)
for city in CITIES:
    resid  = all_test_res[city]['y_true'] - all_test_res[city]['y_pred']
    fitted = all_test_res[city]['y_pred']
    _, p_sw = stats.shapiro(resid[:min(5000,len(resid))])
    _, p_jb = stats.jarque_bera(resid)
    p_lb = acorr_ljungbox(resid,lags=[10],return_df=True)['lb_pvalue'].values[0]
    dw   = durbin_watson(resid)
    _,_,_,p_bp,_ = stats.linregress(fitted, resid**2)
    print('{:<22s} {:>10.4f} {:>10.4f} {:>10.4f} {:>8.3f} {:>10.4f}'.format(
        city,p_sw,p_jb,p_lb,dw,p_bp))
print('\np>0.05 — нет оснований отвергнуть H0; DW≈2 — нет автокорреляции остатков')

Город                    SW p-val   JB p-val   LB p-val       DW   BP p-val
---------------------------------------------------------------------------
Москва                     0.0000     0.0000     0.0000    0.027     0.0000
Санкт-Петербург            0.0000     0.0000     0.0000    0.028     0.0000
Сочи                       0.0000     0.0000     0.0000    0.045     0.0000


Геленджик                  0.0000     0.0000     0.0000    0.027     0.0000
Благовещенск               0.0000     0.0000     0.0000    0.041     0.0000
Находка                    0.0000     0.0000     0.0000    0.058     0.0000

p>0.05 — нет оснований отвергнуть H0; DW≈2 — нет автокорреляции остатков


In [6]:
# Прогноз на месяц вперёд после конца исходных данных
import calendar
last_date = data[CITIES[0]].index[-1]
future_base = forecast_feat_full[CITIES[0]].iloc[-HORIZON:]
future_dates = future_base.index + HORIZON * DATA_STEP
print(f'Прогноз на: {future_dates[0].date()} — {future_dates[-1].date()}')

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
for i, city in enumerate(CITIES):
    fd = forecast_feat_full[city]
    X_last = fd.drop('target',axis=1).iloc[-HORIZON:].values
    Xs = forecast_models[city]['scaler'].transform(X_last)
    preds = forecast_models[city]['model'].predict(Xs)
    hist = data[city]['temperature_2m'].iloc[-60:]
    axes[i].plot(hist.index, hist.values,'b-',lw=1.5,label='История')
    axes[i].plot(future_dates, preds,'r--',lw=1.5,label='Прогноз')
    axes[i].axvline(last_date,color='gray',ls='--',lw=1)
    axes[i].set_title(city,fontsize=11); axes[i].set_ylabel('°C')
    axes[i].legend(fontsize=9); axes[i].grid(alpha=0.3)
    axes[i].xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))
    print(f'{city:22s}: {preds.mean():.1f}°C (ср) [{preds.min():.1f} — {preds.max():.1f}°C]')
fig.suptitle(f'Прогноз температуры: {future_dates[0].strftime("%d.%m.%Y")} — {future_dates[-1].strftime("%d.%m.%Y")}',
             fontsize=13,fontweight='bold')
plt.tight_layout(); plt.show()


Прогноз на: 2026-01-01 — 2026-01-30
Москва                : -8.3°C (ср) [-13.0 — -3.2°C]
Санкт-Петербург       : -5.0°C (ср) [-8.3 — -1.0°C]
Сочи                  : 5.9°C (ср) [2.3 — 12.9°C]
Геленджик             : 4.1°C (ср) [0.7 — 8.9°C]
Благовещенск          : -21.3°C (ср) [-32.4 — -14.4°C]
Находка               : -8.5°C (ср) [-15.5 — -1.1°C]


### Аналитические выводы по разделам 2.5–2.6

1. После пересчёта горизонта как **30 дней = 720 почасовых шагов** ошибка прогноза на тесте 2025 года находится в диапазоне **MAE 2.76-4.53°C** и **RMSE 3.68-5.52°C**.
2. Лучший MAE получен для **Находки** (2.76°C), затем идут **Сочи** (2.92°C), **Геленджик** (3.27°C) и **Благовещенск** (3.44°C). Наиболее сложными для прогноза оказались Москва (4.53°C) и Санкт-Петербург (4.15°C).
3. По R² модели объясняют значительную часть изменчивости температуры: от **0.6246** для Санкт-Петербурга до **0.9151** для Благовещенска.
4. Для температуры в градусах Цельсия обычный MAPE плохо интерпретируется около 0°C, поэтому дополнительно используется **WAPE**. Он составляет 19.1% для Сочи, 21.9% для Геленджика, 22.7% для Находки, 24.3% для Благовещенска, 43.5% для Москвы и 43.7% для Санкт-Петербурга.
5. Тесты остатков показывают p-value Ljung-Box < 0.05 и Durbin-Watson около 0.03-0.06, то есть в ошибках сохраняется выраженная автокорреляция. Это ожидаемо для 30-дневного горизонта и указывает на возможность дальнейшего улучшения за счёт авторегрессионной коррекции или моделей, специально учитывающих структуру остатков.


---
## 2.7. Интеграция двух этапов в единый пайплайн

In [7]:
class WeatherPipeline:
    """
    Двухэтапный пайплайн:
      Этап 1 — TS-классификатор определяет тип климата по последнему окну наблюдений
      Этап 2 — Специализированная регрессионная модель прогнозирует T на 30 дней вперёд
    """
    def __init__(self, clf, ts_features, forecast_models_d, forecast_feat_d,
                 cities, climate_map, label_map, horizon=HORIZON, window=W_CLS):
        self.clf = clf; self.ts_features = ts_features
        self.forecast_models_d = forecast_models_d
        self.forecast_feat_d = forecast_feat_d
        self.cities = cities; self.climate_map = climate_map
        self.inv_label = {v:k for k,v in label_map.items()}
        self.horizon = horizon; self.window = window
        self.climate_to_cities = {}
        for c,cl in climate_map.items(): self.climate_to_cities.setdefault(cl,[]).append(c)

    def classify(self, city_df):
        """Определяет тип климата по последним window наблюдениям."""
        df_norm = (city_df[self.ts_features] - city_df[self.ts_features].mean()) / \
                  (city_df[self.ts_features].std() + 1e-8)
        arr = df_norm.values[-self.window:].T[np.newaxis]  # (1, C, T)
        arr = arr.astype(np.float32)
        pred = int(self.clf.predict(arr)[0])
        try:
            proba = self.clf.predict_proba(arr)[0]
            conf = float(proba[pred])
            proba_d = {self.inv_label[i]: float(p) for i, p in enumerate(proba)}
        except Exception:
            conf = 1.0
            proba_d = {self.inv_label[pred]: 1.0}
        return self.inv_label[pred], conf, proba_d

    def forecast(self, city_name):
        """Прогнозирует температуру для указанного города на заданный горизонт."""
        fd = self.forecast_feat_d[city_name]
        X_last = fd.drop('target',axis=1).iloc[-self.horizon:].values
        Xs = self.forecast_models_d[city_name]['scaler'].transform(X_last)
        preds = self.forecast_models_d[city_name]['model'].predict(Xs)
        step = fd.index.to_series().diff().dropna().median()
        future = fd.index[-self.horizon:] + self.horizon * step
        return pd.Series(preds, index=future, name=f'forecast_{city_name}')

    def run(self, city_df, city_name):
        print(f'=== Пайплайн для {city_name} ===')
        climate, conf, proba = self.classify(city_df)
        expected = self.climate_map[city_name]
        match = 'OK' if climate == expected else 'ОШИБКА'
        print(f'Этап 1 — Климат: {climate} (уверенность {conf:.1%}) [{match}]')
        print(f'         Вероятности: { {k: f"{v:.1%}" for k,v in proba.items()} }')
        fc = self.forecast(city_name)
        print(f'Этап 2 — Прогноз: ср={fc.mean():.1f}°C [{fc.min():.1f}—{fc.max():.1f}°C]')
        return {'climate':climate,'confidence':conf,'forecast':fc}

pipeline = WeatherPipeline(
    clf=best_clf_ts, ts_features=TS_FEATURES,
    forecast_models_d=forecast_models, forecast_feat_d=forecast_feat_full,
    cities=CITIES, climate_map=CLIMATE_MAP, label_map=LABEL_MAP,
    horizon=HORIZON, window=W_CLS
)
print('Пайплайн инициализирован\n')
for city in CITIES:
    pipeline.run(data[city], city); print()

Пайплайн инициализирован

=== Пайплайн для Москва ===
Этап 1 — Климат: умеренно-континентальный (уверенность 100.0%) [OK]
         Вероятности: {'умеренно-континентальный': '100.0%', 'субтропический': '0.0%', 'резко-континентальный': '0.0%'}
Этап 2 — Прогноз: ср=-8.3°C [-13.0—-3.2°C]

=== Пайплайн для Санкт-Петербург ===
Этап 1 — Климат: умеренно-континентальный (уверенность 100.0%) [OK]
         Вероятности: {'умеренно-континентальный': '100.0%', 'субтропический': '0.0%', 'резко-континентальный': '0.0%'}
Этап 2 — Прогноз: ср=-5.0°C [-8.3—-1.0°C]

=== Пайплайн для Сочи ===
Этап 1 — Климат: субтропический (уверенность 100.0%) [OK]
         Вероятности: {'умеренно-континентальный': '0.0%', 'субтропический': '100.0%', 'резко-континентальный': '0.0%'}
Этап 2 — Прогноз: ср=5.9°C [2.3—12.9°C]

=== Пайплайн для Геленджик ===
Этап 1 — Климат: субтропический (уверенность 100.0%) [OK]
         Вероятности: {'умеренно-континентальный': '0.0%', 'субтропический': '100.0%', 'резко-континентальный': 

---
## 2.8. Документирование и интерпретация
### Итоговая сводная таблица

In [8]:
# Сводная таблица
print('='*100)
print('ИТОГОВЫЕ РЕЗУЛЬТАТЫ')
print('='*100)
print(f'\nКЛАССИФИКАЦИЯ (тест 2025, модель: {best_name})')
y_p = ts_test_res[best_name]['y_pred']
print(f'  Accuracy: {accuracy_score(y_te,y_p):.4f}')
print(f'  F1 macro: {f1_score(y_te,y_p,average="macro",zero_division=0):.4f}')
print(f'  F1 wt:    {f1_score(y_te,y_p,average="weighted",zero_division=0):.4f}')
if ts_test_res[best_name]["y_prob"] is not None:
    print(f'  ROC-AUC:  {roc_auc_score(y_te,ts_test_res[best_name]["y_prob"],multi_class="ovr"):.4f}')

print(f'\nПРОГНОЗИРОВАНИЕ (тест 2025, горизонт {HORIZON_DAYS} дней)')
print('{:<22s} {:>12s} {:>7s} {:>7s} {:>7s} {:>7s} {:>7s} {:>8s}'.format(
    'Город','Модель','MAE','RMSE','MAPE*%','WAPE%','R²','DA%'))
print('-'*88)
for city in CITIES:
    r = all_test_res[city]
    mn = {'rf':'RandomForest','ridge':'Ridge','gb':'GB/LGB'}[MODEL_CFG[city]]
    print('{:<22s} {:>12s} {:>7.2f} {:>7.2f} {:>7.1f} {:>7.1f} {:>7.4f} {:>8.1f}'.format(
        city,mn,r['mae'],r['rmse'],r['mape'],r['wape'],r['r2'],r['da']))
print('='*100)

ИТОГОВЫЕ РЕЗУЛЬТАТЫ

КЛАССИФИКАЦИЯ (тест 2025, модель: MiniRocket)
  Accuracy: 1.0000
  F1 macro: 1.0000
  F1 wt:    1.0000
  ROC-AUC:  1.0000

ПРОГНОЗИРОВАНИЕ (тест 2025, горизонт 30 дней)
Город                        Модель     MAE    RMSE  MAPE*%   WAPE%      R²      DA%
----------------------------------------------------------------------------------------
Москва                 RandomForest    4.53    5.52    75.7    43.5  0.6555     64.8
Санкт-Петербург        RandomForest    4.15    5.25    86.5    43.7  0.6246     62.5
Сочи                          Ridge    2.92    3.78    33.9    19.1  0.7602     71.0
Геленджик                     Ridge    3.27    4.14    40.3    21.9  0.7734     71.7
Благовещенск                 GB/LGB    3.44    4.60    44.4    24.3  0.9151     71.7
Находка                      GB/LGB    2.76    3.68    48.9    22.7  0.9016     71.5


In [28]:
# Итоговый дашборд
fig = plt.figure(figsize=(18,12))
gs = GridSpec(3,3,figure=fig,hspace=0.45,wspace=0.35)

# Scatter: прогноз vs факт
ax1 = fig.add_subplot(gs[0,:2])
for i,city in enumerate(CITIES):
    r = all_test_res[city]
    ax1.scatter(r['y_true'],r['y_pred'],c=PALETTE[i],alpha=0.3,s=8,label=city)
lim = [min(all_test_res[c]['y_true'].min() for c in CITIES)-2,
       max(all_test_res[c]['y_true'].max() for c in CITIES)+2]
ax1.plot(lim,lim,'k--',lw=1.5,label='Идеал'); ax1.set_xlim(lim); ax1.set_ylim(lim)
ax1.set_xlabel('Факт (°C)'); ax1.set_ylabel('Прогноз (°C)'); ax1.set_title('Прогноз vs Факт')
ax1.legend(fontsize=8,markerscale=2); ax1.grid(alpha=0.3)

# MAE по городам
ax2 = fig.add_subplot(gs[0,2])
maes = [all_test_res[c]['mae'] for c in CITIES]
brs = ax2.barh(CITIES,maes,color=PALETTE,alpha=0.8)
for bar,val in zip(brs,maes): ax2.text(val+0.05,bar.get_y()+bar.get_height()/2,f'{val:.2f}°C',va='center',fontsize=9)
ax2.set_title('MAE по городам'); ax2.grid(alpha=0.3,axis='x')

# Accuracy TS-классификаторов
if len(ts_test_res)>1:
    ax3 = fig.add_subplot(gs[1,:])
    ns = list(ts_test_res.keys()); accs=[ts_test_res[n]['acc'] for n in ns]; f1s=[ts_test_res[n]['f1m'] for n in ns]
    x=np.arange(len(ns)); w=0.35
    ax3.bar(x-w/2,accs,w,label='Accuracy',color='steelblue',alpha=0.8)
    ax3.bar(x+w/2,f1s,w,label='F1 macro',color='orange',alpha=0.8)
    ax3.set_xticks(x); ax3.set_xticklabels(ns,rotation=15,ha='right')
    ax3.set_ylim(0,1.05); ax3.set_title('Сравнение TS-классификаторов')
    ax3.legend(); ax3.grid(alpha=0.3,axis='y')
else:
    ax3 = fig.add_subplot(gs[1,:])
    ax3.text(0.5,0.5,'Модель: '+str(best_name)+'  Acc='+f'{best_acc:.4f}',ha='center',va='center',
             fontsize=14,transform=ax3.transAxes)
    ax3.set_title('TS-классификатор')

# Остатки
ax4 = fig.add_subplot(gs[2,:])
for i,city in enumerate(CITIES):
    resid = all_test_res[city]['y_true']-all_test_res[city]['y_pred']
    ax4.hist(resid,bins=40,alpha=0.5,color=PALETTE[i],label=city,density=True)
xr = np.linspace(-15,15,200)
ax4.plot(xr,stats.norm.pdf(xr,0,3),'k-',lw=2,label='N(0,3)')
ax4.set_title('Распределение остатков прогнозирования'); ax4.set_xlabel('Ошибка (°C)'); ax4.set_ylabel('Плотность')
ax4.legend(fontsize=9); ax4.grid(alpha=0.3)

fig.suptitle('Итоговый дашборд: классификация и прогнозирование метеоданных',fontsize=14,fontweight='bold')
plt.show()

### Итоговые выводы

**Двухэтапный подход:**
1. **Классификация климата** выполнена специализированными моделями временных рядов ROCKET, MiniRocket, TimeSeriesForest и RISE. Лучшей на тесте 2025 стала **MiniRocket** с Accuracy=1.0000, F1-macro=1.0000 и ROC-AUC=1.0000.
2. **Прогнозирование температуры** построено отдельными моделями для разных климатических зон: RandomForest для Москвы и Санкт-Петербурга, Ridge для Сочи и Геленджика, GB/LGB для Благовещенска и Находки. Горизонт прогноза задан как 30 дней, что для почасовых данных соответствует 720 временным шагам.
3. **Качество прогноза** на горизонте 30 дней: MAE по городам составляет от 2.76°C до 4.53°C, R² — от 0.6246 до 0.9151. Наиболее стабильные относительные ошибки по WAPE получены для Сочи, Геленджика, Находки и Благовещенска.
4. **Единый пайплайн** `WeatherPipeline` объединяет оба этапа: принимает метеоряд, определяет тип климата и формирует прогноз температуры на следующий 30-дневный период.
5. **Ограничения результата:** в остатках сохраняется автокорреляция, поэтому прогноз можно улучшать добавлением авторегрессионной коррекции, более длинной истории или специализированных моделей для последовательностей.
